# Data Preprocessing for Emotion Detection
This notebook preprocesses two datasets for emotion classification:
1. **Empathetic Dialogue** - Uses `empathetic_dialogues` as text, `emotion` as label (32 classes)
2. **Daily Dialogue** - Flattened to utterance-level, with 7 emotion classes

**Output:** Cleaned CSV files ready for model training.

In [ ]:
# Install dependencies (run once)
%pip install pandas scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

## 1. Empathetic Dialogue Dataset

In [ ]:
# Load Empathetic Dialogue
# NOTE: Update path if running on Colab (upload files or use Google Drive)
empathetic_df = pd.read_csv('Datasets/empathetic Dialogue.csv')
print(f"Original shape: {empathetic_df.shape}")
empathetic_df.head()

In [ ]:
# Clean Empathetic Dialogue
# Select relevant columns
empathetic_clean = empathetic_df[['empathetic_dialogues', 'emotion']].copy()

# Drop rows with missing values
empathetic_clean = empathetic_clean.dropna(subset=['empathetic_dialogues', 'emotion'])

# Clean text: remove extra whitespace
empathetic_clean['text'] = empathetic_clean['empathetic_dialogues'].str.strip()
empathetic_clean['text'] = empathetic_clean['text'].str.replace(r'\s+', ' ', regex=True)

# Rename columns for consistency
empathetic_clean = empathetic_clean[['text', 'emotion']].rename(columns={'emotion': 'label'})

print(f"Cleaned shape: {empathetic_clean.shape}")
print(f"\nUnique emotions ({empathetic_clean['label'].nunique()}): {empathetic_clean['label'].unique()}")

In [ ]:
# Encode emotion labels
empathetic_label_encoder = LabelEncoder()
empathetic_clean['label_encoded'] = empathetic_label_encoder.fit_transform(empathetic_clean['label'])

# Save label mapping
empathetic_label_mapping = dict(zip(empathetic_label_encoder.classes_, range(len(empathetic_label_encoder.classes_))))
print("Label Mapping:")
print(empathetic_label_mapping)

In [ ]:
# Filter out classes with too few samples to split stratifiably
# We need enough samples for the 80/10/10 split. 
# To have at least 2 samples in the 20% temp split, we need approx 10 samples total.
class_counts = empathetic_clean['label_encoded'].value_counts()
rare_classes = class_counts[class_counts < 10].index

if len(rare_classes) > 0:
    print(f"Filtering out {len(rare_classes)} rare classes (count < 10): {rare_classes.tolist()}")
    empathetic_filtered = empathetic_clean[~empathetic_clean['label_encoded'].isin(rare_classes)]
else:
    empathetic_filtered = empathetic_clean

# Split into train/val/test (80/10/10)
emp_train, emp_temp = train_test_split(empathetic_filtered, test_size=0.2, random_state=42, stratify=empathetic_filtered['label_encoded'])
emp_val, emp_test = train_test_split(emp_temp, test_size=0.5, random_state=42, stratify=emp_temp['label_encoded'])

print(f"Train: {len(emp_train)}, Val: {len(emp_val)}, Test: {len(emp_test)}")

In [ ]:
# Save processed Empathetic Dialogue
emp_train[['text', 'label_encoded']].to_csv('Datasets/empathetic_train.csv', index=False)
emp_val[['text', 'label_encoded']].to_csv('Datasets/empathetic_val.csv', index=False)
emp_test[['text', 'label_encoded']].to_csv('Datasets/empathetic_test.csv', index=False)

# Save label mapping for later use
pd.DataFrame(list(empathetic_label_mapping.items()), columns=['emotion', 'label_id']).to_csv('Datasets/empathetic_label_map.csv', index=False)
print("Empathetic Dialogue preprocessing complete!")

## 2. Daily Dialogue Dataset

In [ ]:
# Load Daily Dialogue (already has train/val/test splits)
daily_train = pd.read_csv('Datasets/Daily dialouge/train.csv')
daily_val = pd.read_csv('Datasets/Daily dialouge/validation.csv')
daily_test = pd.read_csv('Datasets/Daily dialouge/test.csv')

print(f"Train: {len(daily_train)}, Val: {len(daily_val)}, Test: {len(daily_test)}")
daily_train.head()

In [ ]:
def parse_daily_dialogue(df):
    """
    Flatten Daily Dialogue: each utterance becomes a row with its corresponding emotion.
    
    Emotion mapping:
    0 = no emotion, 1 = anger, 2 = disgust, 3 = fear, 4 = happiness, 5 = sadness, 6 = surprise
    """
    rows = []
    
    for idx, row in df.iterrows():
        try:
            # Parse dialog list
            dialog_str = row['dialog']
            dialog_list = ast.literal_eval(dialog_str)
            
            # Parse emotion array (format: [0 1 2 ...])
            emotion_str = str(row['emotion'])
            # Handle format like '[0 0 0 0]' or '[0, 0, 0, 0]'
            emotion_str = emotion_str.strip('[]')
            emotions = [int(x) for x in emotion_str.replace(',', ' ').split()]
            
            # Match utterances with emotions
            for i, (utterance, emotion) in enumerate(zip(dialog_list, emotions)):
                # Clean utterance
                utterance = utterance.strip()
                if utterance:  # Skip empty utterances
                    rows.append({
                        'text': utterance,
                        'label_encoded': emotion
                    })
        except Exception as e:
            # Skip problematic rows
            continue
    
    return pd.DataFrame(rows)

In [ ]:
# Process each split
daily_train_flat = parse_daily_dialogue(daily_train)
daily_val_flat = parse_daily_dialogue(daily_val)
daily_test_flat = parse_daily_dialogue(daily_test)

print(f"Flattened Train: {len(daily_train_flat)}, Val: {len(daily_val_flat)}, Test: {len(daily_test_flat)}")

In [ ]:
# Check emotion distribution
daily_emotion_map = {
    0: 'no_emotion',
    1: 'anger',
    2: 'disgust',
    3: 'fear',
    4: 'happiness',
    5: 'sadness',
    6: 'surprise'
}

print("Train emotion distribution:")
print(daily_train_flat['label_encoded'].value_counts().sort_index())
print("\nNote: Class 0 (no_emotion) is typically dominant. We'll use class weights during training.")

In [ ]:
# Save processed Daily Dialogue
daily_train_flat.to_csv('Datasets/daily_train.csv', index=False)
daily_val_flat.to_csv('Datasets/daily_val.csv', index=False)
daily_test_flat.to_csv('Datasets/daily_test.csv', index=False)

# Save label mapping
pd.DataFrame(list(daily_emotion_map.items()), columns=['label_id', 'emotion']).to_csv('Datasets/daily_label_map.csv', index=False)
print("Daily Dialogue preprocessing complete!")

## 3. Summary

In [ ]:
print("=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print("\nEmpathetic Dialogue:")
print(f"  - Train: {len(emp_train)} samples")
print(f"  - Val: {len(emp_val)} samples")
print(f"  - Test: {len(emp_test)} samples")
print(f"  - Classes: {len(empathetic_label_mapping)}")
print("\nDaily Dialogue:")
print(f"  - Train: {len(daily_train_flat)} utterances")
print(f"  - Val: {len(daily_val_flat)} utterances")
print(f"  - Test: {len(daily_test_flat)} utterances")
print(f"  - Classes: 7")
print("\nOutput files saved to 'Datasets/' folder.")